In [1]:
import os
os.chdir("/home/mdarif/MLOPS/Wine-Quality-Prediction")
print(os.getcwd())

/home/mdarif/MLOPS/Wine-Quality-Prediction


In [2]:
import pandas as pd
data = pd.read_csv("artifacts/data_ingestion/WineQT.csv")
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,4


In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    source_file: Path
    schema: dict
    status_file: Path

In [4]:
from wine_quality_prediction.constants import *
from wine_quality_prediction.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:

    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH,
            schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        create_directories([config.root_dir])
        return DataValidationConfig(
            root_dir=Path(config.root_dir),
            source_file=Path(config.source_file),
            schema= self.schema.COLUMNS,
            status_file=Path(config.status_file)
        )

In [6]:
import pandas as pd
from wine_quality_prediction.logger import get_logger, DataValidationError
from wine_quality_prediction.utils import get_size, save_status

logger = get_logger("data_validation", "data_validation.log")

In [7]:
class DataValidation:
    """Handles data validation checks."""

    def __init__(self, config: DataValidationConfig):
        self.config = config

    def load_data(self) -> pd.DataFrame:
        """Load data from CSV file."""

        try:
            logger.info(f"Reading file: '{self.config.source_file}'")
            logger.info(f"File size: '{get_size(self.config.source_file)}'")
            df = pd.read_csv(self.config.source_file)
            logger.info(f"Data loaded successfully with shape: {df.shape}'")
            logger.info(f"Columns found: '{df.columns.tolist()}'")
            return df

        except Exception as e:
            logger.error(
                f"Load data failed - path='{self.config.source_file}': {e}",
                exc_info=True
            )
            raise DataValidationError("Failed to load data") from e

    def validate_schema(self, df: pd.DataFrame) -> bool:
        """Check all expected columns exist."""

        try:
            expected_columns = list(self.config.schema.keys())
            missing_columns = [
                col for col in expected_columns
                if col not in df.columns
            ]
            if missing_columns:
                logger.error(
                    f"Schema validation failed - "
                    f"missing columns: '{missing_columns}'"
                )
                return False
            logger.info("Schema validation passed.")
            return True

        except Exception as e:
            logger.error(
                f"Schema validation error: {e}",
                exc_info=True
            )
            raise DataValidationError("Schema validation failed") from e

    def validate_dtypes(self, df: pd.DataFrame) -> bool:
        """Validate column data types."""

        try:
            all_passed = True
            for col, expected_dtype in self.config.schema.items():
                if col in df.columns:
                    actual_dtype = str(df[col].dtype)
                    if actual_dtype != expected_dtype:
                        logger.error(
                            f"Dtype mismatch - col='{col}' "
                            f"expected='{expected_dtype}' "
                            f"got='{actual_dtype}'"
                        )
                        all_passed = False
            if all_passed:
                logger.info("Data type validation passed.")
            return all_passed

        except Exception as e:
            logger.error(
                f"Data type validation error: {e}",
                exc_info=True
            )
            raise DataValidationError("Data type validation failed") from e

    def validate_missing_values(self, df: pd.DataFrame) -> bool:
        """Check for null/NaN values."""

        try:
            null_count = df.isnull().sum()
            col_with_nulls = null_count[null_count > 0]
            if len(col_with_nulls) > 0:
                logger.error(
                    f"Missing value validation failed - "
                    f"null values found:\n'{col_with_nulls}'"
                )
                return False
            logger.info("Missing value validation passed.")
            return True

        except Exception as e:
            logger.error(
                f"Missing value validation error: {e}",
                exc_info=True
            )
            raise DataValidationError("Missing value validation failed") from e

    def validate_value_range(self, df: pd.DataFrame) -> bool:
        """Check value range for key columns."""

        try:
            all_passed = True
            ranges = {
                "quality": (0, 10),
                "pH": (0, 14),
                "alcohol": (0, 100),
                "density": (0, 2)
            }
            for col, (min_val, max_val) in ranges.items():
                if col in df.columns:
                    out_of_range = df[(df[col] < min_val) | df[col] > max_val]
                    if len(out_of_range) > 0:
                        logger.error(
                            f"Range validation failed - col='{col}' "
                            f"range='{(min_val, max_val)}' "
                            f"violations='{len(out_of_range)}'"
                        )
                        all_passed = False
            if all_passed:
                logger.info("Value range validation passed.")
            return all_passed

        except Exception as e:
            logger.error(
                f"Value range validation error: {e}",
                exc_info=True
            )
            raise DataValidationError("Value range validation failed") from e

    def validate_duplicates(self, df: pd.DataFrame) -> bool:
        """Check duplicate rows."""

        try:
            duplicate_count = df.duplicated().sum()
            if duplicate_count > 0:
                logger.warning(
                    f"Duplicate validation failed - found '{duplicate_count}' duplicate rows"
                )
                return False
            logger.info("Duplicate validation passed.")
            return True

        except Exception as e:
            logger.error(
                f"Duplicate validation error: {e}",
                exc_info=True
            )
            raise DataValidationError("Duplicate validation failed") from e

    def run(self):
        """Run full data validation pipeline."""

        try:
            df = self.load_data()
            schema_check = self.validate_schema(df)
            dtype_check = self.validate_dtypes(df)
            missing_check = self.validate_missing_values(df)
            range_check = self.validate_value_range(df)
            duplicate_check = self.validate_duplicates(df)
            overall_status = all([
                schema_check,
                dtype_check,
                missing_check,
                range_check,
                duplicate_check
            ])
            save_status(
                self.config.status_file,
                "data_validation",
                overall_status
            )
            if not overall_status:
                logger.error("Data validation failed.")
            else:
                logger.info("Data validation passed.")

        except Exception as e:
            save_status(
                self.config.status_file,
                "data_validation",
                False
            )
            logger.error(
                f"Unexpected error: {e}",
                exc_info=True
            )
            raise DataValidationError("Data validation pipeline failed") from e

In [8]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.run()
except Exception:
    raise

2026-06-02 08:17:22,181 - common - INFO - common.py:31 - YAML file loaded successfully: 'config/config.yaml'
2026-06-02 08:17:22,183 - common - INFO - common.py:31 - YAML file loaded successfully: 'params.yaml'
2026-06-02 08:17:22,186 - common - INFO - common.py:31 - YAML file loaded successfully: 'schema.yaml'
2026-06-02 08:17:22,188 - common - INFO - common.py:53 - Created directory at: 'artifacts'
2026-06-02 08:17:22,189 - common - INFO - common.py:53 - Created directory at: 'artifacts/data_validation'
2026-06-02 08:17:22,190 - data_validation - INFO - 3490742956.py:11 - Reading file: 'artifacts/data_ingestion/WineQT.csv'
2026-06-02 08:17:22,192 - data_validation - INFO - 3490742956.py:12 - File size: '~76 KB'
2026-06-02 08:17:22,196 - data_validation - INFO - 3490742956.py:14 - Data loaded successfully with shape: (1143, 13)'
2026-06-02 08:17:22,197 - data_validation - INFO - 3490742956.py:15 - Columns found: '['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', '